# Лабораторная работа №11: Локальные малые модели и дистилляция

## Цель работы
Изучить работу с малыми языковыми моделями (SLM — Small Language Models), методы дистилляции знаний от больших моделей к малым, и сравнение эффективности различных подходов.

## Теоретическая часть

### Что такое Small Language Models?
SLM — это компактные модели с количеством параметров от нескольких миллионов до нескольких миллиардов:
- **Преимущества**: быстрый инференс, низкие требования к памяти, возможность запуска на CPU
- **Недостатки**: меньшая способность к обобщению, ограниченные знания

### Популярные SLM
- **TinyLlama** (1.1B) — компактная версия Llama
- **Phi-3** (3.8B) — от Microsoft, высокое качество при малом размере
- **Qwen2.5** (0.5B-7B) — семейство моделей от Alibaba
- **Gemma** (2B-7B) — от Google
- **DistilBERT** — дистиллированная версия BERT

### Методы дистилляции

#### 1. Knowledge Distillation (Hinton et al.)
- Student модель обучается имитировать output teacher модели
- Используется temperature scaling для смягчения вероятностей

#### 2. Task-specific Distillation
- Дистилляция для конкретной задачи (классификация, QA)
- Более эффективна для узких доменов

#### 3. Self-distillation
- Модель обучается на своих собственных предсказаниях
- Итеративное улучшение качества

#### 4. DistilLLM подходы
- **MiniLLM** — генеративная дистилляция
- **GKD** — generalized knowledge distillation

## Практическая часть

### Установка зависимостей

In [ ]:
!pip install transformers torch accelerate sentencepiece -q
!pip install datasets evaluate scikit-learn -q

### Импорт библиотек

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
import time
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Сравнение различных малых моделей

In [ ]:
# Список моделей для сравнения
models_to_compare = {
    "TinyLlama-1.1B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen2.5-0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "Phi-3-mini": "microsoft/phi-3-mini-4k-instruct"
}

# Тестовый промпт
test_prompts = [
    "Объясни простыми словами, что такое машинное обучение.",
    "Напиши функцию на Python для вычисления чисел Фибоначчи.",
    "Какие основные преимущества использования Docker?"
]

def compare_models(models, prompts, max_new_tokens=100):
    """Сравнивает скорость и качество моделей."""
    results = []
    
    for model_name, model_path in models.items():
        print(f"\n{'='*60}")
        print(f"Тестирование модели: {model_name}")
        print(f"{'='*60}")
        
        try:
            # Загрузка токенизатора и модели
            print("Загрузка токенизатора...")
            tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
            
            print("Загрузка модели...")
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
            
            # Подсчет параметров
            num_params = sum(p.numel() for p in model.parameters()) / 1e6
            print(f"Количество параметров: {num_params:.1f}M")
            
            model_results = {
                'name': model_name,
                'params_millions': num_params,
                'responses': []
            }
            
            # Тестирование на каждом промпте
            for i, prompt in enumerate(prompts, 1):
                print(f"\nПромпт {i}: {prompt[:50]}...")
                
                inputs = tokenizer(prompt, return_tensors="pt")
                if torch.cuda.is_available():
                    inputs = {k: v.cuda() for k, v in inputs.items()}
                
                # Измерение времени
                start_time = time.time()
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id
                    )
                
                end_time = time.time()
                inference_time = end_time - start_time
                
                response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                generated_text = response[len(prompt):].strip()
                
                print(f"Время генерации: {inference_time:.2f} сек")
                print(f"Ответ: {generated_text[:150]}..." if len(generated_text) > 150 else f"Ответ: {generated_text}")
                
                model_results['responses'].append({
                    'prompt': prompt,
                    'response': generated_text,
                    'inference_time': inference_time
                })
            
            results.append(model_results)
            
            # Очистка памяти
            del model, tokenizer
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"Ошибка при тестировании {model_name}: {e}")
            results.append({
                'name': model_name,
                'error': str(e)
            })
    
    return results

# Запуск сравнения (раскомментировать для выполнения)
# results = compare_models(models_to_compare, test_prompts)

### Дистилляция знаний: практический пример

Рассмотрим дистилляцию BERT в меньшую модель для задачи классификации

In [ ]:
# Teacher модель (больше)
teacher_model_name = "distilbert-base-uncased"  # Для примера используем DistilBERT как teacher

# Student модель (меньше) - создадим упрощенную архитектуру
from transformers import BertConfig

student_config = BertConfig(
    vocab_size=30522,
    hidden_size=256,      # Меньше чем у teacher (768)
    num_hidden_layers=4,  # Меньше чем у teacher (6)
    num_attention_heads=4,# Меньше чем у teacher (12)
    intermediate_size=512,# Меньше чем у teacher (3072)
    max_position_embeddings=512
)

print(f"Конфигурация Student модели:")
print(f"  Hidden size: {student_config.hidden_size}")
print(f"  Layers: {student_config.num_hidden_layers}")
print(f"  Attention heads: {student_config.num_attention_heads}")

num_params_student = (
    student_config.num_hidden_layers * (
        4 * student_config.hidden_size**2 +  # FFN
        4 * student_config.hidden_size * student_config.hidden_size +  # Attention
        2 * student_config.hidden_size  # LayerNorm
    ) + student_config.vocab_size * student_config.hidden_size  # Embeddings
) / 1e6

print(f"\nОриентировочное количество параметров: {num_params_student:.1f}M")

### Простая реализация Knowledge Distillation

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class KnowledgeDistillationLoss(nn.Module):
    """Loss функция для knowledge distillation."""
    
    def __init__(self, temperature=2.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha  # Вес для distillation loss
        self.kl_div = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()
    
    def forward(self, student_logits, teacher_logits, labels):
        # Soft targets (с temperature scaling)
        soft_targets = F.softmax(teacher_logits / self.temperature, dim=-1)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        
        # Distillation loss (KL divergence)
        distill_loss = self.kl_div(soft_student, soft_targets) * (self.temperature ** 2)
        
        # Task loss (Cross entropy с истинными лейблами)
        task_loss = self.ce(student_logits, labels)
        
        # Комбинированный loss
        loss = self.alpha * distill_loss + (1 - self.alpha) * task_loss
        
        return loss, distill_loss.item(), task_loss.item()

print("Knowledge Distillation Loss создан!")
print(f"Temperature: 2.0, Alpha: 0.7")

### Загрузка датасета для тренировки

In [ ]:
# Загрузка небольшого датасета для классификации
dataset = load_dataset("imdb", split="train[:1000]")  # Первые 1000 примеров

print(f"Размер датасета: {len(dataset)} примеров")
print(f"Колонки: {dataset.column_names}")
print(f"\nПример текста:\n{dataset[0]['text'][:200]}...")
print(f"Лейбл: {dataset[0]['label']} (0=negative, 1=positive)")

### Задание 1: Сравнение производительности SLM

Запустите сравнение 2-3 малых моделей:
1. Измерьте время инференса
2. Оцените качество ответов субъективно
3. Сравните использование памяти
4. Сделайте выводы о применимости для разных задач

In [ ]:
# Ваш код здесь
# Раскомментируйте и запустите compare_models()

### Задание 2: Реализация полной дистилляции

Реализуйте полный цикл дистилляции:
1. Загрузите teacher модель
2. Создайте student модель
3. Реализуйте тренировочный цикл с KD loss
4. Обучите student на части датасета
5. Сравните качество student и teacher

In [ ]:
# Ваш код здесь
# Реализуйте полный цикл knowledge distillation

### Задание 3: Оптимизация SLM для деплоя

Исследуйте методы оптимизации малых моделей:
1. Квантование (INT8, INT4)
2. Прунинг (pruning)
3. Экспорт в ONNX
4. Сравнение скорости до и после оптимизации

In [ ]:
# Ваш код здесь
# Реализуйте оптимизацию SLM

## Контрольные вопросы

1. В каких случаях целесообразно использовать SLM вместо больших моделей?
2. Как temperature scaling влияет на процесс дистилляции?
3. Какие проблемы могут возникнуть при дистилляции и как их решать?
4. Чем отличается distillation от обычного fine-tuning?
5. Как оценить эффективность дистилляции?

## Требования к отчету

1. Таблица сравнения производительности различных SLM
2. Описание реализованного метода дистилляции
3. Графики обучения student модели
4. Сравнение качества teacher и student моделей
5. Результаты оптимизации SLM
6. Ответы на контрольные вопросы
7. Выводы о применимости SLM для различных задач